# 03. Feature engineering cho đề 18

Bài toán chính của nhóm là hồi quy dự đoán `G3`. Notebook này chuẩn bị biến đầu vào và tạo hai kịch bản để thực nghiệm:

- Scenario A: có sử dụng `G1`, `G2` và tạo thêm đặc trưng từ hai điểm này.
- Scenario B: không sử dụng `G1`, `G2`, phù hợp với dự đoán sớm.

Các phép biến đổi ở đây chỉ là biến đổi xác định được từ từng dòng. Encoder và scaler sẽ được fit riêng trên train ở notebook huấn luyện.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

def find_project_root():
    current = Path.cwd().resolve()
    for folder in [current, *current.parents]:
        if (folder / 'data').exists() and (folder / 'src').exists():
            return folder
    raise FileNotFoundError('Không tìm thấy thư mục gốc của project.')

PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = ARTIFACTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
valid_df = pd.read_csv(PROCESSED_DIR / 'valid.csv')
test_df = pd.read_csv(PROCESSED_DIR / 'test.csv')
print(f'Train: {train_df.shape}, valid: {valid_df.shape}, test: {test_df.shape}')

In [ ]:
def build_features(frame, use_previous_grades=True):
    '''Tạo các feature dùng chung cho cả train, valid, test và API.'''.strip()
    result = frame.copy()

    # Log giúp giảm ảnh hưởng của các giá trị absences quá lớn.
    result['absences_log1p'] = np.log1p(result['absences'])

    if use_previous_grades:
        result['previous_grade_mean'] = (result['G1'] + result['G2']) / 2
        result['previous_grade_gap'] = result['G2'] - result['G1']
    else:
        result = result.drop(columns=['G1', 'G2'])

    return result

def make_regression_data(frame, use_previous_grades=True):
    engineered = build_features(frame, use_previous_grades)
    X = engineered.drop(columns=['G3'])
    y = engineered['G3'].astype(float)
    return X, y

def make_classification_data(frame, use_previous_grades=True):
    engineered = build_features(frame, use_previous_grades)
    X = engineered.drop(columns=['G3'])
    y = (engineered['G3'] >= 10).astype(int)
    return X, y

In [ ]:
scenarios = {
    'A_with_G1_G2': True,
    'B_without_G1_G2': False
}
feature_sets = {}
scenario_rows = []
scenario_metadata = {}

for scenario_name, use_previous_grades in scenarios.items():
    X_train, y_train = make_regression_data(train_df, use_previous_grades)
    X_valid, y_valid = make_regression_data(valid_df, use_previous_grades)
    X_test, y_test = make_regression_data(test_df, use_previous_grades)
    feature_sets[scenario_name] = {
        'X_train': X_train, 'y_train': y_train,
        'X_valid': X_valid, 'y_valid': y_valid,
        'X_test': X_test, 'y_test': y_test
    }
    scenario_rows.append({
        'scenario': scenario_name,
        'use_G1_G2': use_previous_grades,
        'feature_count_before_encoding': X_train.shape[1],
        'numeric_features': X_train.select_dtypes(include='number').columns.tolist(),
        'categorical_features': X_train.select_dtypes(exclude='number').columns.tolist()
    })
    scenario_metadata[scenario_name] = {
        'use_G1_G2': use_previous_grades,
        'feature_columns': X_train.columns.tolist()
    }

scenario_table = pd.DataFrame(scenario_rows)
display(scenario_table[['scenario', 'use_G1_G2', 'feature_count_before_encoding']])
assert all('G3' not in item['feature_columns'] for item in scenario_metadata.values())
(ARTIFACTS_DIR / 'feature_scenarios.json').write_text(
    json.dumps(scenario_metadata, ensure_ascii=False, indent=2), encoding='utf-8'
)

In [ ]:
# Xem các feature mới được tạo trong Scenario A.
display(feature_sets['A_with_G1_G2']['X_train'][['G1', 'G2', 'previous_grade_mean', 'previous_grade_gap', 'absences', 'absences_log1p']].head())

# Kiểm tra tỷ lệ đạt/trượt cho phần mở rộng classification.
class_balance = pd.concat({
    'train': train_df.assign(passed=(train_df['G3'] >= 10)),
    'valid': valid_df.assign(passed=(valid_df['G3'] >= 10)),
    'test': test_df.assign(passed=(test_df['G3'] >= 10))
}).groupby(['passed']).size().rename('rows').to_frame()
display(class_balance)

fig, ax = plt.subplots(figsize=(6, 4))
plot_train = train_df.assign(passed=np.where(train_df['G3'] >= 10, 'passed', 'failed'))
sns.countplot(data=plot_train, x='passed', ax=ax)
ax.set_title('Phân bố nhãn đạt/trượt trên train')
ax.set_xlabel('Nhãn')
ax.set_ylabel('Số dòng')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'feature_pass_fail_balance.png', dpi=150)
plt.show()

## Cách dùng kết quả

Notebook 04 sẽ dùng lại đúng hàm `build_features` và fit bộ tiền xử lý riêng cho từng scenario. Khi nhóm viết `src/features/build_features.py`, cần đưa logic ở notebook này vào module chính thức để API xử lý input mới giống hệt lúc train.